# SQL Agent (SQLite Demo) — Step-by-Step Walkthrough

**Blauw-Zwart Analytics Pipeline**

This notebook walks through the SQL agent pipeline using a **self-contained
SQLite database** — no Docker, Postgres, or Kafka required. A full season of
synthetic match-day and retail events is generated from `match_day.example.json`
and loaded into a compact SQLite schema that mirrors the generator-backed dbt
fan marts.

An English-language question enters, a validated SQL query is generated and
executed, and a Markdown answer comes back — all without the user ever writing
SQL.

This notebook now creates the dbt-style marts `mart_fan_loyalty` and
`mart_match_summary`. The player mart is intentionally left out here.

---

### Pipeline at a glance

```
User question
      │
      ▼
┌─────────────────────────────────────────────────────┐
│            SQLite Demo Agent  (ReAct loop)          │
│                                                     │
│  list_tables → describe_table → search_columns      │
│  → sample_table → execute_select ✓                  │
│                                                     │
│  ┌──────────────────────────────────────────────┐   │
│  │         SQL Guardrails (3 layers)            │   │
│  │  1. strip_fences()                           │   │
│  │  2. rewrite_schema_qualifiers()              │   │
│  │  3. validate_sql()  (sqlglot AST + regex)    │   │
│  └──────────────────────────────────────────────┘   │
└──────────────────────┬──────────────────────────────┘
                       │
                       ▼
               SQLite (local file)
               LIMIT 100 safety net
                       │
                       ▼
               Markdown answer
```

### Steps covered in this notebook

| Step | What you'll see |
|------|----------------|
| **0. Setup** | Generate full-season match-day + retail data and create dbt-aligned SQLite marts |
| **1. Schema discovery** | `list_tables`, `describe_table`, `search_columns`, `sample_table` |
| **2. SQL guardrails** | Fence stripping, schema rewriting, AST + regex validation |
| **3. Safe execution** | `execute_select` through the full guardrail pipeline |
| **4. Full walkthrough** | End-to-end agent-like tool-call sequence |
| **5. ReAct agent** | *(optional, needs API key)* Live agent answering a question |

**Key libraries:**
| Library | Role |
|---------|------|
| `langchain` / `langgraph` | ReAct agent loop and tool-calling framework |
| `sqlglot` | SQL parsing, AST validation, and schema rewriting |
| `sqlite3` | Lightweight local database (stdlib — no install needed) |
| `fan_events.generation.v2_calendar` | Synthetic match-day event generation |
| `fan_events.generation.v3_retail` | Synthetic retail event generation |

---
## 0 · Setup — Generate Data & Create SQLite Database

Generate a full season of synthetic fan activity from the 30-match calendar in
`match_day.example.json`, then combine it with synthetic retail purchases so the
SQLite demo mirrors the generator-backed dbt fan-event layer more closely.

The setup creates a compact production-inspired schema:

- `match_events`
- `merch_purchase`
- `retail_purchase`
- `match_metadata`
- `mart_fan_loyalty`
- `mart_match_summary`
- `mart_dbt_run_results` *(empty placeholder schema — dbt hooks do not run in this notebook)*

`mart_player_season_summary` is intentionally omitted in this notebook.

We use a tiny on-disk database instead of `:memory:` so the later ReAct tools
can open fresh connections safely across worker threads.

**No Docker required** — this runs entirely on the standard library + repo code.

In [59]:
import json
import random
import sqlite3
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk upwards until we find the repository root markers."""
    markers = ("pyproject.toml", "dbt_project.yml", "docker-compose.yml")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory."
    )


def _open_sqlite_connection(database_path: Path) -> sqlite3.Connection:
    """Open a fresh connection to the demo SQLite database.

    Using one short-lived connection per tool call keeps the notebook compatible
    with LangGraph's thread-pooled tool execution.

    Args:
        database_path: Path to the SQLite database file.

    Returns:
        Configured SQLite connection with row access by column name.
    """
    connection = sqlite3.connect(database_path)
    connection.row_factory = sqlite3.Row
    return connection


def _create_base_tables(cursor: sqlite3.Cursor) -> None:
    """Create the compact production-inspired SQLite schema.

    The notebook mirrors the generator-backed fan-event side of the dbt project
    with helper tables plus two dbt-style marts. `mart_dbt_run_results` is
    created as an empty placeholder because this notebook does not execute dbt
    hooks.

    Args:
        cursor: Cursor bound to the demo SQLite database.
    """
    cursor.executescript(
        """
        CREATE TABLE match_events (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            ingested_at TEXT    NOT NULL,
            event_type  TEXT    NOT NULL,
            event_time  TEXT    NOT NULL,
            timestamp   TEXT    NOT NULL,
            fan_id      TEXT    NOT NULL,
            match_id    TEXT    NOT NULL,
            amount      REAL,
            item        TEXT,
            location    TEXT
        );

        CREATE TABLE retail_purchase (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            ingested_at TEXT    NOT NULL,
            event_time  TEXT    NOT NULL,
            timestamp   TEXT    NOT NULL,
            fan_id      TEXT    NOT NULL,
            shop        TEXT    NOT NULL,
            item        TEXT    NOT NULL,
            amount      REAL    NOT NULL
        );

        CREATE TABLE match_metadata (
            match_id                               TEXT PRIMARY KEY,
            kickoff_local                          TEXT,
            timezone                               TEXT,
            attendance                             INTEGER,
            home_away                              TEXT,
            encounter_type                         TEXT,
            opponent                               TEXT,
            home_score                             INTEGER,
            away_score                             INTEGER,
            venue_label                            TEXT,
            club_home_club                         TEXT,
            club_home_stadium                      TEXT,
            club_home_stadium_capacity             INTEGER,
            club_home_reported_total_attendance    INTEGER,
            club_home_reported_average_attendance  INTEGER,
            club_home_reported_home_matches        INTEGER,
            club_home_reported_sold_out_matches    INTEGER,
            club_home_reported_capacity_pct        REAL
        );

        CREATE TABLE mart_dbt_run_results (
            invocation_id    TEXT,
            run_started_at   TEXT,
            logged_at        TEXT,
            project_name     TEXT,
            target_name      TEXT,
            node_unique_id   TEXT,
            node_name        TEXT,
            resource_type    TEXT,
            materialization  TEXT,
            schema_name      TEXT,
            relation_name    TEXT,
            status           TEXT,
            is_success       INTEGER,
            is_failure       INTEGER,
            execution_time_s REAL,
            rows_affected    INTEGER,
            failures         INTEGER,
            message          TEXT
        );
        """
    )


def _materialize_marts(cursor: sqlite3.Cursor) -> None:
    """Build helper tables and dbt-style marts inside SQLite.

    Args:
        cursor: Cursor bound to the demo SQLite database.
    """
    # First derive the match-linked merch helper table directly from match_events.
    cursor.execute(
        """
        CREATE TABLE merch_purchase AS
        SELECT
            id,
            ingested_at,
            event_time,
            timestamp,
            fan_id,
            match_id,
            item,
            amount,
            location
        FROM match_events
        WHERE event_type = 'merch_purchase'
        """
    )

    # Then build the fan mart using the same logical ingredients as dbt:
    # merch purchases, retail purchases, and ticket scans.
    cursor.execute(
        """
        CREATE TABLE mart_fan_loyalty AS
        WITH fan_ids AS (
            SELECT fan_id FROM merch_purchase
            UNION
            SELECT fan_id FROM retail_purchase
            UNION
            SELECT fan_id FROM match_events
        ),
        merch_agg AS (
            SELECT
                fan_id,
                COUNT(id) AS merch_purchase_count,
                COALESCE(SUM(amount), 0.0) AS merch_total_spend,
                MAX(ingested_at) AS last_ingested_at
            FROM merch_purchase
            GROUP BY fan_id
        ),
        merch_item_counts AS (
            SELECT fan_id, item, COUNT(*) AS purchase_count
            FROM merch_purchase
            WHERE item IS NOT NULL AND item != ''
            GROUP BY fan_id, item
        ),
        merch_item_rank AS (
            SELECT fan_id, item
            FROM (
                SELECT
                    fan_id,
                    item,
                    ROW_NUMBER() OVER (
                        PARTITION BY fan_id
                        ORDER BY purchase_count DESC, item ASC
                    ) AS rn
                FROM merch_item_counts
            )
            WHERE rn = 1
        ),
        retail_agg AS (
            SELECT
                fan_id,
                COUNT(id) AS retail_purchase_count,
                COALESCE(SUM(amount), 0.0) AS retail_total_spend,
                MAX(ingested_at) AS last_ingested_at
            FROM retail_purchase
            GROUP BY fan_id
        ),
        retail_item_counts AS (
            SELECT fan_id, item, COUNT(*) AS purchase_count
            FROM retail_purchase
            WHERE item IS NOT NULL AND item != ''
            GROUP BY fan_id, item
        ),
        retail_item_rank AS (
            SELECT fan_id, item
            FROM (
                SELECT
                    fan_id,
                    item,
                    ROW_NUMBER() OVER (
                        PARTITION BY fan_id
                        ORDER BY purchase_count DESC, item ASC
                    ) AS rn
                FROM retail_item_counts
            )
            WHERE rn = 1
        ),
        shop_counts AS (
            SELECT fan_id, shop, COUNT(*) AS purchase_count
            FROM retail_purchase
            WHERE shop IS NOT NULL AND shop != ''
            GROUP BY fan_id, shop
        ),
        shop_rank AS (
            SELECT fan_id, shop
            FROM (
                SELECT
                    fan_id,
                    shop,
                    ROW_NUMBER() OVER (
                        PARTITION BY fan_id
                        ORDER BY purchase_count DESC, shop ASC
                    ) AS rn
                FROM shop_counts
            )
            WHERE rn = 1
        ),
        attendance_agg AS (
            SELECT
                fan_id,
                COUNT(DISTINCT match_id) AS matches_attended,
                MIN(event_time) AS first_match_attended,
                MAX(event_time) AS last_match_attended,
                MAX(ingested_at) AS last_ingested_at
            FROM match_events
            WHERE event_type = 'ticket_scan'
            GROUP BY fan_id
        )
        SELECT
            fans.fan_id,
            COALESCE(ma.merch_purchase_count, 0) AS merch_purchase_count,
            ROUND(COALESCE(ma.merch_total_spend, 0.0), 2) AS merch_total_spend,
            mir.item AS favourite_merch_item,
            COALESCE(ra.retail_purchase_count, 0) AS retail_purchase_count,
            ROUND(COALESCE(ra.retail_total_spend, 0.0), 2) AS retail_total_spend,
            rir.item AS favourite_retail_item,
            sr.shop AS favourite_shop,
            ROUND(
                COALESCE(ma.merch_total_spend, 0.0)
                + COALESCE(ra.retail_total_spend, 0.0),
                2
            ) AS total_spend,
            COALESCE(aa.matches_attended, 0) AS matches_attended,
            aa.first_match_attended,
            aa.last_match_attended,
            MAX(
                COALESCE(ma.last_ingested_at, '1970-01-01T00:00:00Z'),
                COALESCE(ra.last_ingested_at, '1970-01-01T00:00:00Z'),
                COALESCE(aa.last_ingested_at, '1970-01-01T00:00:00Z')
            ) AS last_updated_at
        FROM fan_ids AS fans
        LEFT JOIN merch_agg AS ma ON fans.fan_id = ma.fan_id
        LEFT JOIN merch_item_rank AS mir ON fans.fan_id = mir.fan_id
        LEFT JOIN retail_agg AS ra ON fans.fan_id = ra.fan_id
        LEFT JOIN retail_item_rank AS rir ON fans.fan_id = rir.fan_id
        LEFT JOIN shop_rank AS sr ON fans.fan_id = sr.fan_id
        LEFT JOIN attendance_agg AS aa ON fans.fan_id = aa.fan_id
        ORDER BY total_spend DESC, fans.fan_id ASC
        """
    )

    # Finally build the match mart from match metadata, match-linked events,
    # and aggregated stadium merch revenue.
    cursor.execute(
        """
        CREATE TABLE mart_match_summary AS
        WITH event_agg AS (
            SELECT
                match_id,
                COUNT(id) AS match_event_count,
                SUM(CASE WHEN event_type = 'ticket_scan' THEN 1 ELSE 0 END) AS ticket_scan_count,
                COUNT(DISTINCT CASE WHEN event_type = 'ticket_scan' THEN fan_id END)
                    AS ticket_scanned_fans,
                MIN(event_time) AS first_event_time,
                MAX(event_time) AS last_event_time,
                MAX(ingested_at) AS last_ingested_at
            FROM match_events
            GROUP BY match_id
        ),
        merch_agg AS (
            SELECT
                match_id,
                COUNT(id) AS merch_purchase_count,
                COUNT(DISTINCT fan_id) AS unique_merch_buyers,
                COALESCE(SUM(amount), 0.0) AS merch_revenue,
                MAX(ingested_at) AS last_ingested_at
            FROM merch_purchase
            GROUP BY match_id
        ),
        merch_item_counts AS (
            SELECT match_id, item, COUNT(*) AS purchase_count
            FROM merch_purchase
            WHERE item IS NOT NULL AND item != ''
            GROUP BY match_id, item
        ),
        merch_item_rank AS (
            SELECT match_id, item
            FROM (
                SELECT
                    match_id,
                    item,
                    ROW_NUMBER() OVER (
                        PARTITION BY match_id
                        ORDER BY purchase_count DESC, item ASC
                    ) AS rn
                FROM merch_item_counts
            )
            WHERE rn = 1
        )
        SELECT
            b.match_id,
            b.kickoff_local,
            b.timezone,
            b.home_away,
            b.encounter_type,
            b.opponent,
            b.venue_label,
            b.home_score,
            b.away_score,
            CASE
                WHEN b.home_away = 'home' THEN b.home_score
                ELSE b.away_score
            END AS club_brugge_score,
            CASE
                WHEN b.home_away = 'home' THEN b.away_score
                ELSE b.home_score
            END AS opponent_score,
            CASE
                WHEN (
                    CASE
                        WHEN b.home_away = 'home' THEN b.home_score
                        ELSE b.away_score
                    END
                ) > (
                    CASE
                        WHEN b.home_away = 'home' THEN b.away_score
                        ELSE b.home_score
                    END
                ) THEN 'win'
                WHEN (
                    CASE
                        WHEN b.home_away = 'home' THEN b.home_score
                        ELSE b.away_score
                    END
                ) < (
                    CASE
                        WHEN b.home_away = 'home' THEN b.away_score
                        ELSE b.home_score
                    END
                ) THEN 'loss'
                ELSE 'draw'
            END AS match_result,
            b.attendance AS reported_attendance,
            COALESCE(e.match_event_count, 0) AS match_event_count,
            COALESCE(e.ticket_scan_count, 0) AS ticket_scan_count,
            COALESCE(e.ticket_scanned_fans, 0) AS ticket_scanned_fans,
            CASE
                WHEN b.attendance IS NOT NULL AND b.attendance > 0 THEN
                    ROUND(100.0 * COALESCE(e.ticket_scan_count, 0) / b.attendance, 1)
            END AS ticket_scan_pct_of_reported_attendance,
            COALESCE(m.merch_purchase_count, 0) AS merch_purchase_count,
            COALESCE(m.unique_merch_buyers, 0) AS unique_merch_buyers,
            ROUND(COALESCE(m.merch_revenue, 0.0), 2) AS merch_revenue,
            mir.item AS top_merch_item,
            b.club_home_club,
            b.club_home_stadium,
            b.club_home_stadium_capacity,
            b.club_home_reported_total_attendance,
            b.club_home_reported_average_attendance,
            b.club_home_reported_home_matches,
            b.club_home_reported_sold_out_matches,
            b.club_home_reported_capacity_pct,
            CASE
                WHEN b.home_away = 'home'
                 AND b.club_home_stadium_capacity IS NOT NULL
                 AND b.club_home_stadium_capacity > 0
                 AND b.attendance IS NOT NULL THEN
                    ROUND(100.0 * b.attendance / b.club_home_stadium_capacity, 1)
            END AS home_capacity_utilization_pct,
            CASE
                WHEN b.home_away = 'home'
                 AND b.club_home_stadium_capacity IS NOT NULL
                 AND b.attendance IS NOT NULL THEN
                    b.attendance >= b.club_home_stadium_capacity
            END AS is_home_capacity_sellout,
            e.first_event_time,
            e.last_event_time,
            MAX(
                COALESCE(e.last_ingested_at, '1970-01-01T00:00:00Z'),
                COALESCE(m.last_ingested_at, '1970-01-01T00:00:00Z')
            ) AS last_updated_at
        FROM match_metadata AS b
        LEFT JOIN event_agg AS e ON b.match_id = e.match_id
        LEFT JOIN merch_agg AS m ON b.match_id = m.match_id
        LEFT JOIN merch_item_rank AS mir ON b.match_id = mir.match_id
        ORDER BY b.kickoff_local, b.match_id
        """
    )


# ── Resolve paths and make project importable ──────────────────────────
REPO_ROOT = _find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
SQLITE_DB_PATH = REPO_ROOT / "notebooks" / "fan_events.db"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"✅ Repo root: {REPO_ROOT}")
print("   src/ added to sys.path")
print(f"   SQLite file: {SQLITE_DB_PATH}")

# ── Load the match calendar and generator helpers ──────────────────────
from fan_events.generation.orchestrator import (
    default_unified_fan_pool_max,
    iter_merged_records,
)
from fan_events.generation.retail_intensity import build_retail_rate_factor_fn
from fan_events.generation.v2_calendar import (
    filter_matches_by_date_range,
    iter_v2_records_merged_sorted,
    load_calendar_json,
    validate_and_parse_matches,
)
from fan_events.generation.v3_retail import iter_retail_records

calendar_path = REPO_ROOT / "match_day.example.json"
doc = load_calendar_json(calendar_path)
parsed_matches = validate_and_parse_matches(doc)
print(f"✅ Loaded {len(parsed_matches)} matches from {calendar_path.name}")

# ── Generate a merged calendar + retail stream like the Compose stack ──
seed = 42
calendar_rng = random.Random(f"sqlite-demo:v2:{seed}")
retail_rng = random.Random(f"sqlite-demo:retail:{seed}")
contexts = filter_matches_by_date_range(parsed_matches, from_date=None, to_date=None)
shared_fan_pool_max = default_unified_fan_pool_max(contexts)
season_start_utc = min(ctx.window_start for ctx in contexts)
season_end_utc = max(ctx.window_end for ctx in contexts)
season_duration_seconds = max((season_end_utc - season_start_utc).total_seconds(), 1.0)
target_retail_events = max(1_500, len(contexts) * 60)
base_poisson_rate = max(target_retail_events / season_duration_seconds / 2.0, 1e-6)
retail_rate_factor_fn = build_retail_rate_factor_fn(
    contexts,
    home_match_day_multiplier=2.0,
    home_kickoff_pre_minutes=90,
    home_kickoff_post_minutes=120,
    home_kickoff_extra_multiplier=1.5,
    away_match_day_enable=False,
    away_match_day_multiplier=1.75,
)

v2_iter = iter_v2_records_merged_sorted(
    contexts,
    calendar_rng,
    fan_pool_max=shared_fan_pool_max,
)
retail_iter = iter_retail_records(
    retail_rng,
    epoch_utc=season_start_utc,
    max_simulated_duration_seconds=season_duration_seconds,
    poisson_rate=base_poisson_rate,
    fan_pool=shared_fan_pool_max,
    rate_factor_fn=retail_rate_factor_fn,
)
records = list(iter_merged_records(retail_iter, v2_iter))
match_records = [rec for rec in records if rec.get("match_id")]
retail_records = [rec for rec in records if rec.get("event") == "retail_purchase"]
print(f"✅ Generated {len(records):,} merged events across {len(contexts)} matches")
print(f"   shared fan pool: {shared_fan_pool_max:,}")
print(f"   target retail events: ~{target_retail_events:,}")

# ── Create the demo SQLite database file ───────────────────────────────
# Close a leftover connection from a previous run so the setup cell stays rerunnable.
previous_db = globals().get("db")
if isinstance(previous_db, sqlite3.Connection):
    previous_db.close()

# Rebuild the database from scratch on each run to keep later examples deterministic.
if SQLITE_DB_PATH.exists():
    SQLITE_DB_PATH.unlink()

db = _open_sqlite_connection(SQLITE_DB_PATH)
cur = db.cursor()
_create_base_tables(cur)

# Load compact event-level helper tables first; marts are built from these.
match_event_rows = [
    (
        rec["timestamp"],
        rec["event"],
        rec["timestamp"],
        rec["timestamp"],
        rec["fan_id"],
        rec["match_id"],
        rec.get("amount"),
        rec.get("item"),
        rec.get("location", ""),
    )
    for rec in match_records
]
cur.executemany(
    """
    INSERT INTO match_events (
        ingested_at,
        event_type,
        event_time,
        timestamp,
        fan_id,
        match_id,
        amount,
        item,
        location
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    match_event_rows,
)

retail_rows = [
    (
        rec["timestamp"],
        rec["timestamp"],
        rec["timestamp"],
        rec["fan_id"],
        rec["shop"],
        rec["item"],
        rec["amount"],
    )
    for rec in retail_records
]
cur.executemany(
    """
    INSERT INTO retail_purchase (
        ingested_at,
        event_time,
        timestamp,
        fan_id,
        shop,
        item,
        amount
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    retail_rows,
)

match_metadata_rows = [
    (
        ctx.row["match_id"],
        ctx.row["kickoff_local"],
        ctx.row["timezone"],
        ctx.row["attendance"],
        ctx.row["home_away"],
        ctx.row.get("encounter_type", ctx.row["home_away"]),
        ctx.row.get("opponent"),
        ctx.row.get("home_score"),
        ctx.row.get("away_score"),
        ctx.row["venue_label"],
        ctx.row.get("club_home_club"),
        ctx.row.get("club_home_stadium"),
        ctx.row.get("club_home_stadium_capacity"),
        ctx.row.get("club_home_reported_total_attendance"),
        ctx.row.get("club_home_reported_average_attendance"),
        ctx.row.get("club_home_reported_home_matches"),
        ctx.row.get("club_home_reported_sold_out_matches"),
        ctx.row.get("club_home_reported_capacity_pct"),
    )
    for ctx in contexts
]
cur.executemany(
    """
    INSERT INTO match_metadata (
        match_id,
        kickoff_local,
        timezone,
        attendance,
        home_away,
        encounter_type,
        opponent,
        home_score,
        away_score,
        venue_label,
        club_home_club,
        club_home_stadium,
        club_home_stadium_capacity,
        club_home_reported_total_attendance,
        club_home_reported_average_attendance,
        club_home_reported_home_matches,
        club_home_reported_sold_out_matches,
        club_home_reported_capacity_pct
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    match_metadata_rows,
)

_materialize_marts(cur)
db.commit()

# ── Summary stats ──────────────────────────────────────────────────────
n_events = cur.execute("SELECT COUNT(*) FROM match_events").fetchone()[0]
n_fans = cur.execute("SELECT COUNT(*) FROM mart_fan_loyalty").fetchone()[0]
n_matches = cur.execute("SELECT COUNT(*) FROM mart_match_summary").fetchone()[0]
n_scans = cur.execute(
    "SELECT COUNT(*) FROM match_events WHERE event_type='ticket_scan'"
).fetchone()[0]
n_merch = cur.execute(
    "SELECT COUNT(*) FROM merch_purchase"
).fetchone()[0]
n_retail = cur.execute(
    "SELECT COUNT(*) FROM retail_purchase"
).fetchone()[0]

print("\n✅ SQLite database ready with dbt-aligned fan marts")
print(f"   match_events:         {n_events:,} rows ({n_scans:,} scans + {n_merch:,} merch)")
print(f"   retail_purchase:     {n_retail:,} rows")
print(f"   mart_fan_loyalty:    {n_fans:,} unique fans")
print(f"   mart_match_summary:  {n_matches:,} matches")
print("   mart_dbt_run_results: 0 rows (placeholder schema)")

# Quick peeks at both marts so the new tables are immediately visible.
print("\n   Sample from mart_fan_loyalty (top 3 by spend):")
for row in cur.execute(
    """
    SELECT fan_id, total_spend, retail_total_spend, matches_attended
    FROM mart_fan_loyalty
    ORDER BY total_spend DESC, fan_id ASC
    LIMIT 3
    """
).fetchall():
    print(f"     {dict(row)}")

print("\n   Sample from mart_match_summary (top 3 by merch revenue):")
for row in cur.execute(
    """
    SELECT match_id, opponent, merch_revenue, ticket_scan_count
    FROM mart_match_summary
    ORDER BY merch_revenue DESC, match_id ASC
    LIMIT 3
    """
).fetchall():
    print(f"     {dict(row)}")

print("\n   Sample from match_events (first 3 rows):")
for row in cur.execute(
    """
    SELECT id, event_type, fan_id, match_id, amount, item
    FROM match_events
    ORDER BY id ASC
    LIMIT 3
    """
).fetchall():
    print(f"     {dict(row)}")

print("\n   Sample from merch_purchase (first 3 rows):")
for row in cur.execute(
    """
    SELECT id, fan_id, match_id, item, amount
    FROM merch_purchase
    ORDER BY id ASC
    LIMIT 3
    """
).fetchall():
    print(f"     {dict(row)}")

print("\n   Sample from retail_purchase (first 3 rows):")
for row in cur.execute(
    """
    SELECT id, fan_id, shop, item, amount
    FROM retail_purchase
    ORDER BY id ASC
    LIMIT 3
    """
).fetchall():
    print(f"     {dict(row)}")

print("\n   Sample from match_metadata (first 3 rows):")
for row in cur.execute(
    """
    SELECT match_id, kickoff_local, home_away, opponent, attendance, home_score, away_score
    FROM match_metadata
    ORDER BY kickoff_local ASC
    LIMIT 3
    """
).fetchall():
    print(f"     {dict(row)}")

✅ Repo root: D:\Projecten_Thuis\blauw_zwart_fan_sim_pipeline
   src/ added to sys.path
   SQLite file: D:\Projecten_Thuis\blauw_zwart_fan_sim_pipeline\notebooks\fan_events.db
✅ Loaded 30 matches from match_day.example.json
✅ Generated 616,573 merged events across 30 matches
   shared fan pool: 29,062
   target retail events: ~1,800

✅ SQLite database ready with dbt-aligned fan marts
   match_events:         615,670 rows (475,745 scans + 139,925 merch)
   retail_purchase:     903 rows
   mart_fan_loyalty:    29,062 unique fans
   mart_match_summary:  30 matches
   mart_dbt_run_results: 0 rows (placeholder schema)

   Sample from mart_fan_loyalty (top 3 by spend):
     {'fan_id': 'fan_16383', 'total_spend': 477.12, 'retail_total_spend': 0.0, 'matches_attended': 17}
     {'fan_id': 'fan_28738', 'total_spend': 457.16, 'retail_total_spend': 0.0, 'matches_attended': 16}
     {'fan_id': 'fan_19862', 'total_spend': 449.86, 'retail_total_spend': 0.0, 'matches_attended': 18}

   Sample from mart

---
## 1 · Schema Discovery Tools

The production SQL agent discovers the database schema on demand using
read-only tools. Below we build **SQLite equivalents** of each discovery
tool so we can demonstrate the same workflow without Postgres.

| Tool | Production (Postgres) | This notebook (SQLite) |
|------|----------------------|----------------------|
| `list_tables()` | `information_schema.tables` | `sqlite_master` |
| `describe_table(t)` | `information_schema.columns` | `PRAGMA table_info(t)` |
| `search_columns(p)` | `column_name ILIKE %p%` | iterate PRAGMAs + Python filter |
| `sample_table(t, n)` | `SELECT * FROM t LIMIT n` | same |

In [50]:
import re

_VALID_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def _fetch_all_sqlite(sql: str) -> list[sqlite3.Row]:
    """Execute one SQLite query against the demo database and return all rows.

    Args:
        sql: SQL statement to execute.

    Returns:
        Result rows with dict-style column access.
    """
    conn = _open_sqlite_connection(SQLITE_DB_PATH)
    try:
        return conn.execute(sql).fetchall()
    finally:
        conn.close()


def list_tables() -> list[dict[str, str]]:
    """List all tables in the SQLite database.

    Returns:
        One dictionary per table with `name` and `type` keys.
    """
    rows = _fetch_all_sqlite(
        "SELECT name, type FROM sqlite_master "
        "WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' "
        "ORDER BY name"
    )
    return [{"name": r["name"], "type": r["type"]} for r in rows]


def describe_table(table: str) -> dict:
    """Describe one table: column name, type, nullable, and primary-key flag.

    Args:
        table: Table name to inspect.

    Returns:
        Dictionary with the table name and a `columns` list, or an `error`
        message when the identifier is invalid or unknown.
    """
    if not _VALID_IDENT.match(table):
        return {"error": f"Invalid identifier: {table!r}"}

    known = {t["name"] for t in list_tables()}
    if table not in known:
        return {
            "error": f"Unknown table {table!r}. Call list_tables() to see options."
        }

    rows = _fetch_all_sqlite(f"PRAGMA table_info({table})")
    columns = []
    for r in rows:
        columns.append(
            {
                "name": r["name"],
                "data_type": r["type"] or "ANY",
                "nullable": r["notnull"] == 0,
                "pk": bool(r["pk"]),
            }
        )
    return {"name": table, "columns": columns}


def search_columns(pattern: str) -> list[dict[str, str]]:
    """Search column names across all tables.

    Args:
        pattern: Case-insensitive substring to match against column names.

    Returns:
        Matching `(table, column, data_type)` triples.
    """
    pat = pattern.lower()
    results = []

    # Reuse the public helpers so the demo mirrors the agent's discovery flow.
    for tbl in list_tables():
        desc = describe_table(tbl["name"])
        for col in desc.get("columns", []):
            if pat in col["name"].lower():
                results.append(
                    {
                        "table": tbl["name"],
                        "column": col["name"],
                        "data_type": col["data_type"],
                    }
                )
    return results


def sample_table(table: str, limit: int = 5) -> list[dict]:
    """Return a small sample of rows from one table.

    Args:
        table: Table name to sample.
        limit: Requested row count, capped to keep outputs compact.

    Returns:
        List of row dictionaries, or a one-item error list for invalid input.
    """
    if not _VALID_IDENT.match(table):
        return [{"error": f"Invalid identifier: {table!r}"}]

    known = {t["name"] for t in list_tables()}
    if table not in known:
        return [{"error": f"Unknown table {table!r}. Call list_tables() to see options."}]

    cap = max(1, min(limit, 10))
    rows = _fetch_all_sqlite(f"SELECT * FROM {table} LIMIT {cap}")
    return [dict(r) for r in rows]


print("✅ SQLite discovery tools defined: list_tables, describe_table, search_columns, sample_table")

✅ SQLite discovery tools defined: list_tables, describe_table, search_columns, sample_table


In [51]:
# ── Example usage of list_tables() ─────────────────────────────
print("=" * 60)
print("TOOL: list_tables()")
print("=" * 60)
print("\nThe agent's first step — discover what tables exist.\n")

tables = list_tables()
print(f"Found {len(tables)} tables in SQLite:\n")
print(f"  {'Name':<20} {'Type'}")
print(f"  {'-' * 20} {'-' * 10}")
for t in tables:
    print(f"  {t['name']:<20} {t['type']}")

TOOL: list_tables()

The agent's first step — discover what tables exist.

Found 7 tables in SQLite:

  Name                 Type
  -------------------- ----------
  mart_dbt_run_results table
  mart_fan_loyalty     table
  mart_match_summary   table
  match_events         table
  match_metadata       table
  merch_purchase       table
  retail_purchase      table


In [52]:
# ── Example usage of describe_table() ─────────────────────────
print("=" * 60)
print("TOOL: describe_table('mart_fan_loyalty')")
print("=" * 60)
print("\nThe agent inspects columns, types, and constraints before writing SQL.\n")

desc = describe_table("mart_fan_loyalty")
print(f"  Table: {desc['name']}\n")
print(f"  {'Column':<25} {'Type':<10} {'Nullable':<10} {'PK'}")
print(f"  {'-' * 25} {'-' * 10} {'-' * 10} {'-' * 5}")
for col in desc["columns"]:
    print(f"  {col['name']:<25} {col['data_type']:<10} {str(col['nullable']):<10} {col['pk']}")

print("\n" + "=" * 60)
print("TOOL: describe_table('mart_match_summary')")
print("=" * 60 + "\n")

desc2 = describe_table("mart_match_summary")
print(f"  Table: {desc2['name']}\n")
print(f"  {'Column':<25} {'Type':<10} {'Nullable':<10} {'PK'}")
print(f"  {'-' * 25} {'-' * 10} {'-' * 10} {'-' * 5}")
for col in desc2["columns"]:
    print(f"  {col['name']:<25} {col['data_type']:<10} {str(col['nullable']):<10} {col['pk']}")

TOOL: describe_table('mart_fan_loyalty')

The agent inspects columns, types, and constraints before writing SQL.

  Table: mart_fan_loyalty

  Column                    Type       Nullable   PK
  ------------------------- ---------- ---------- -----
  fan_id                    TEXT       True       False
  merch_purchase_count      ANY        True       False
  merch_total_spend         ANY        True       False
  favourite_merch_item      TEXT       True       False
  retail_purchase_count     ANY        True       False
  retail_total_spend        ANY        True       False
  favourite_retail_item     TEXT       True       False
  favourite_shop            TEXT       True       False
  total_spend               ANY        True       False
  matches_attended          ANY        True       False
  first_match_attended      ANY        True       False
  last_match_attended       ANY        True       False
  last_updated_at           ANY        True       False

TOOL: describe_table(

In [53]:
# ── Example usage of search_columns() ─────────────────────────
print("=" * 60)
print("TOOL: search_columns('spend')")
print("=" * 60)
print("\nSearch column names across ALL tables (case-insensitive).\n")

results = search_columns("spend")
print(f"  Found {len(results)} columns matching 'spend':\n")
print(f"  {'Table':<20} {'Column':<25} {'Type'}")
print(f"  {'-' * 20} {'-' * 25} {'-' * 10}")
for r in results:
    print(f"  {r['table']:<20} {r['column']:<25} {r['data_type']}")

# Also search for 'fan' to show cross-table results
print("\n" + "=" * 60)
print("TOOL: search_columns('fan')")
print("=" * 60 + "\n")

results2 = search_columns("fan")
print(f"  Found {len(results2)} columns matching 'fan':\n")
for r in results2:
    print(f"  {r['table']:<20} {r['column']:<25} {r['data_type']}")

TOOL: search_columns('spend')

Search column names across ALL tables (case-insensitive).

  Found 3 columns matching 'spend':

  Table                Column                    Type
  -------------------- ------------------------- ----------
  mart_fan_loyalty     merch_total_spend         ANY
  mart_fan_loyalty     retail_total_spend        ANY
  mart_fan_loyalty     total_spend               ANY

TOOL: search_columns('fan')

  Found 5 columns matching 'fan':

  mart_fan_loyalty     fan_id                    TEXT
  mart_match_summary   ticket_scanned_fans       ANY
  match_events         fan_id                    TEXT
  merch_purchase       fan_id                    TEXT
  retail_purchase      fan_id                    TEXT


In [54]:
# ── Example usage of sample_table() ─────────────────────────
print("=" * 60)
print("TOOL: sample_table('mart_fan_loyalty', limit=5)")
print("=" * 60)
print("\nPeek at actual rows to verify data shape.\n")

sample = sample_table("mart_fan_loyalty", limit=5)
if sample:
    cols = list(sample[0].keys())
    header = "  " + " | ".join(f"{c:<20}" for c in cols)
    print(header)
    print("  " + "-" * len(header))
    for row in sample:
        vals = " | ".join(f"{str(row.get(c, '')):<20}" for c in cols)
        print(f"  {vals}")

print("\n" + "=" * 60)
print("TOOL: sample_table('match_events', limit=3)")
print("=" * 60 + "\n")

sample2 = sample_table("match_events", limit=3)
if sample2:
    cols2 = list(sample2[0].keys())
    header2 = "  " + " | ".join(f"{c:<15}" for c in cols2)
    print(header2)
    print("  " + "-" * len(header2))
    for row in sample2:
        vals = " | ".join(f"{str(row.get(c, '')):<15}" for c in cols2)
        print(f"  {vals}")

TOOL: sample_table('mart_fan_loyalty', limit=5)

Peek at actual rows to verify data shape.

  fan_id               | merch_purchase_count | merch_total_spend    | favourite_merch_item | retail_purchase_count | retail_total_spend   | favourite_retail_item | favourite_shop       | total_spend          | matches_attended     | first_match_attended | last_match_attended  | last_updated_at     
  ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  fan_16383            | 17                   | 477.12               | Beanie 1891 navy     | 0                    | 0.0                  | None                 | None                 | 477.12               | 17                   | 2025-07-27T14:39:47Z | 2026-03-22T18:54:13Z | 2026-03-22T18:54:13Z
  fan

---
## 2 · SQL Guardrails

Before any SQL reaches the database it passes through **three layers** of
sanitisation (all in `guardrails.py` — imported directly, they are backend-agnostic):

```
Raw LLM output
      │
      ▼
┌─────────────────────────────────────┐
│  1. _strip_fences()                 │  Remove ```sql ... ``` wrappers
├─────────────────────────────────────┤
│  2. _rewrite_layer_schema_qualifiers│  marts.X → dbt_dev.X
├─────────────────────────────────────┤
│  3. _validate_sql()                 │  Two-pass validation:
│     ├─ sqlglot AST check            │   parse → reject DDL/DML nodes
│     └─ Regex belt-and-braces        │   catch mutating keywords
└─────────────────────────────────────┘
      │
      ▼
  Clean, validated SQL → database
```

In [55]:
from frontend_app.sql_agent.guardrails import (
    _rewrite_layer_schema_qualifiers,
    _strip_fences,
    _validate_sql,
)

# ── Layer 1: Strip code fences ─────────────────────────────────────
print("=" * 60)
print("GUARDRAIL 1: _strip_fences()")
print("=" * 60)
print("\nLLMs often wrap SQL in markdown code fences.\n")

# Test sql query with fences + semicolon (common LLM output)
raw_from_llm = """```sql
SELECT fan_id, total_spend
FROM marts.mart_fan_loyalty
ORDER BY total_spend DESC
LIMIT 5;
```"""

print("  INPUT (raw LLM output):")
for line in raw_from_llm.strip().split("\n"):
    print(f"    {line}")

step1 = _strip_fences(raw_from_llm)
print(f"\n  OUTPUT (fences + trailing semicolons removed):")
for line in step1.strip().split("\n"):
    print(f"    {line}")

# ── Layer 2: Rewrite schema qualifiers ────────────────────────────
print("\n" + "=" * 60)
print("GUARDRAIL 2: _rewrite_layer_schema_qualifiers()")
print("=" * 60)
print("\nRewrites marts.X → dbt_dev.X (smaller LLMs confuse layer names with schemas).\n")

step2 = _rewrite_layer_schema_qualifiers(step1)
print("  INPUT:  FROM marts.mart_fan_loyalty")
print("  OUTPUT: FROM dbt_dev.mart_fan_loyalty")
print(f"\n  Full rewritten SQL:")
for line in step2.strip().split("\n"):
    print(f"    {line}")

# ── Layer 3: Validate SQL (AST + regex) ──────────────────────────
print("\n" + "=" * 60)
print("GUARDRAIL 3a: _validate_sql() — SAFE query passes")
print("=" * 60)

safe_sql = "SELECT fan_id, total_spend FROM mart_fan_loyalty ORDER BY total_spend DESC"
print(f"\n  SQL: {safe_sql}")
print(f"\n  Running validation…")

try:
    _validate_sql(safe_sql)
    print("    ├─ sqlglot parse:     ✅ parsed OK")
    print("    ├─ AST node check:    ✅ clean")
    print("    └─ Regex check:       ✅ clean")
    print("\n  ✅ Query passed all guardrails — safe to execute")
except ValueError as exc:
    print(f"  ❌ {exc}")

# ── Layer 3b: Dangerous queries blocked ─────────────────────────
print("\n" + "=" * 60)
print("GUARDRAIL 3b: _validate_sql() — DANGEROUS queries blocked")
print("=" * 60)
print("\nEach is rejected by the AST check, the regex check, or both.\n")

dangerous = [
    ("DROP TABLE mart_fan_loyalty", "DDL"),
    ("SELECT * FROM fans; DELETE FROM fans", "Multi-statement + DML"),
    ("INSERT INTO fans VALUES (1, 'hacker')", "DML"),
    ("UPDATE fans SET name='pwned' WHERE 1=1", "DML"),
    ("TRUNCATE TABLE mart_fan_loyalty", "DDL"),
]

for sql, category in dangerous:
    print(f"  SQL: {sql}")
    print(f"  Category: {category}")
    try:
        _validate_sql(sql)
        print("  Result: ⚠️  PASSED (unexpected!)")
    except ValueError as exc:
        print("  Result: 🛡️  BLOCKED")
        print(f"  Reason: {exc}")
    print()

GUARDRAIL 1: _strip_fences()

LLMs often wrap SQL in markdown code fences.

  INPUT (raw LLM output):
    ```sql
    SELECT fan_id, total_spend
    FROM marts.mart_fan_loyalty
    ORDER BY total_spend DESC
    LIMIT 5;
    ```

  OUTPUT (fences + trailing semicolons removed):
    SELECT fan_id, total_spend
    FROM marts.mart_fan_loyalty
    ORDER BY total_spend DESC
    LIMIT 5

GUARDRAIL 2: _rewrite_layer_schema_qualifiers()

Rewrites marts.X → dbt_dev.X (smaller LLMs confuse layer names with schemas).

  INPUT:  FROM marts.mart_fan_loyalty
  OUTPUT: FROM dbt_dev.mart_fan_loyalty

  Full rewritten SQL:
    SELECT fan_id, total_spend
    FROM dbt_dev.mart_fan_loyalty
    ORDER BY total_spend DESC
    LIMIT 5

GUARDRAIL 3a: _validate_sql() — SAFE query passes

  SQL: SELECT fan_id, total_spend FROM mart_fan_loyalty ORDER BY total_spend DESC

  Running validation…
    ├─ sqlglot parse:     ✅ parsed OK
    ├─ AST node check:    ✅ clean
    └─ Regex check:       ✅ clean

  ✅ Query passed 

---
## 3 · Safe SQL Execution — `execute_select_sqlite`

This is the **only path** for running model-generated SQL. It chains all
three guardrail layers, wraps the query in an outer `LIMIT 100`, and
executes against SQLite.

The function returns structured JSON — identical contract to the production
`execute_select` tool:
- **Success**: `{"rows": [...], "row_count": N, "sql": "..."}`
- **Validation error**: `{"error": "...", "phase": "validation", "sql": "..."}`
- **Execution error**: `{"error": "...", "phase": "execution", "sql": "..."}`

In [56]:
import time


def execute_select_sqlite(sql: str) -> dict:
    """Execute a validated read-only SELECT against SQLite with full guardrails.

    Applies the same three-layer pipeline as production:
    1. Strip markdown fences
    2. Rewrite layer schema qualifiers
    3. Validate SQL (sqlglot AST + regex)
    Then wraps in LIMIT 100 and executes.

    Args:
        sql: SQL text produced by the model or entered manually.

    Returns:
        Dict containing rows plus metadata on success, or a structured error
        payload with `phase` and `sql` keys on failure.
    """
    if not isinstance(sql, str) or not sql.strip():
        return {"error": "sql must be a non-empty string", "phase": "validation"}

    # Guardrail layer 1 + 2: strip fences and rewrite schema prefixes.
    cleaned = _rewrite_layer_schema_qualifiers(_strip_fences(sql))

    # SQLite keeps the demo tables in the default schema, so strip any
    # leftover dbt_dev. prefixes after the production rewrite step.
    cleaned = re.sub(r"\bdbt_dev\.", "", cleaned)

    # Guardrail layer 3: sqlglot AST + regex validation.
    try:
        _validate_sql(cleaned)
    except ValueError as exc:
        return {"error": str(exc), "phase": "validation", "sql": cleaned}

    # Open a short-lived connection here so LangGraph can call this tool from
    # worker threads without reusing the notebook's main-thread connection.
    wrapped = f"SELECT * FROM ({cleaned}) LIMIT 100"

    t0 = time.perf_counter()
    conn = _open_sqlite_connection(SQLITE_DB_PATH)
    try:
        rows = [dict(r) for r in conn.execute(wrapped).fetchall()]
    except Exception as exc:
        return {"error": str(exc), "phase": "execution", "sql": cleaned}
    finally:
        conn.close()
    elapsed_ms = (time.perf_counter() - t0) * 1000

    return {
        "rows": rows,
        "row_count": len(rows),
        "sql": cleaned,
        "elapsed_ms": round(elapsed_ms),
    }


# ── Demo: valid query ─────────────────────────────────────────────
print("=" * 60)
print("execute_select_sqlite() — valid query")
print("=" * 60)

sql = (
    "SELECT fan_id, total_spend, matches_attended "
    "FROM mart_fan_loyalty ORDER BY total_spend DESC LIMIT 5"
)
print(f"\n  SQL: {sql}")
print("  Pipeline: strip_fences → rewrite_schema → validate_sql → execute")

result = execute_select_sqlite(sql)

if "error" in result:
    print(f"\n  ❌ {result['phase']}: {result['error']}")
else:
    print("\n  ✅ Query succeeded!")
    print(f"     Rows returned: {result['row_count']}")
    print(f"     Elapsed:       {result['elapsed_ms']} ms")
    print("\n     Top 5 fans by spend:")
    print(f"     {'fan_id':<15} {'total_spend':>12} {'matches_attended':>18}")
    print(f"     {'-' * 47}")
    for row in result["rows"][:5]:
        print(f"     {row['fan_id']:<15} {row['total_spend']:>12.2f} {row['matches_attended']:>18}")

# ── Demo: blocked query ───────────────────────────────────────────
print("\n" + "=" * 60)
print("execute_select_sqlite() — blocked query")
print("=" * 60)

bad_sql = "DROP TABLE mart_fan_loyalty"
print(f"\n  SQL: {bad_sql}")
result_bad = execute_select_sqlite(bad_sql)
print(f"  Phase:  {result_bad['phase']}")
print(f"  Error:  {result_bad['error']}")
print("\n  ✅ Dangerous query was blocked before reaching the database!")

# ── Demo: LLM schema mistake auto-corrected ──────────────────────
print("\n" + "=" * 60)
print("execute_select_sqlite() — LLM schema mistake auto-corrected")
print("=" * 60)

fenced_sql = """```sql
SELECT fan_id, total_spend FROM marts.mart_fan_loyalty ORDER BY total_spend DESC LIMIT 3
```"""
print("\n  SQL (raw from LLM): marts.mart_fan_loyalty wrapped in fences")
result_fixed = execute_select_sqlite(fenced_sql)

if "error" in result_fixed:
    print(f"  ❌ {result_fixed['phase']}: {result_fixed['error']}")
else:
    print(f"  ✅ Auto-corrected! Executed: {result_fixed['sql']}")
    print(f"     Rows: {result_fixed['row_count']}")

execute_select_sqlite() — valid query

  SQL: SELECT fan_id, total_spend, matches_attended FROM mart_fan_loyalty ORDER BY total_spend DESC LIMIT 5
  Pipeline: strip_fences → rewrite_schema → validate_sql → execute

  ✅ Query succeeded!
     Rows returned: 5
     Elapsed:       13 ms

     Top 5 fans by spend:
     fan_id           total_spend   matches_attended
     -----------------------------------------------
     fan_16383             477.12                 17
     fan_28738             457.16                 16
     fan_19862             449.86                 18
     fan_13105             447.40                 19
     fan_14708             439.99                 19

execute_select_sqlite() — blocked query

  SQL: DROP TABLE mart_fan_loyalty
  Phase:  validation
  Error:  Generated SQL must begin with SELECT or WITH. Received: 'DROP TABLE mart_fan_loyalty'

  ✅ Dangerous query was blocked before reaching the database!

execute_select_sqlite() — LLM schema mistake auto-corrected


---
## 4 · Full Tool Walkthrough

This cell simulates the agent's natural discovery flow — the same sequence of
tool calls the ReAct loop would make when answering a question like:

> *"Who are the top 3 fans by total spend, and how many matches did each attend?"*

```
list_tables → describe_table → search_columns → sample_table → execute_select
```

In [57]:
question = "Who are the top 3 fans by total spend, and how many matches did each attend?"

print("=" * 60)
print("AGENT TOOL-CALL WALKTHROUGH")
print("=" * 60)
print(f"\n  Question: {question}")
print("\n  Simulating the agent's ReAct reasoning steps…\n")

# Step 1: Discover tables
print("─" * 60)
print("Step 1 → list_tables()")
tables = list_tables()
table_names = [t["name"] for t in tables]
print(f"  Agent sees: {table_names}")
print("  Reasoning: mart_fan_loyalty looks like the right table for fan spend data\n")

# Step 2: Inspect mart_fan_loyalty
print("─" * 60)
print("Step 2 → describe_table('mart_fan_loyalty')")
desc = describe_table("mart_fan_loyalty")
col_names = [c["name"] for c in desc["columns"]]
print(f"  Agent sees columns: {col_names}")
print("  Reasoning: total_spend and matches_attended are exactly what I need\n")

# Step 3: Verify with search_columns
print("─" * 60)
print("Step 3 → search_columns('spend')")
spend_cols = search_columns("spend")
for r in spend_cols:
    print(f"  {r['table']}.{r['column']} ({r['data_type']})")
print(
    "  Reasoning: total_spend, merch_total_spend, and retail_total_spend live in"
    " mart_fan_loyalty — confirmed\n"
)

# Step 4: Peek at sample rows
print("─" * 60)
print("Step 4 → sample_table('mart_fan_loyalty', limit=3)")
sample = sample_table("mart_fan_loyalty", limit=3)
for row in sample:
    print(f"  {row}")
print("  Reasoning: data looks reasonable, ready to write the final query\n")

# Step 5: Execute the query
print("─" * 60)
print("Step 5 → execute_select_sqlite()")
final_sql = """
SELECT fan_id, total_spend, matches_attended
FROM mart_fan_loyalty
ORDER BY total_spend DESC
LIMIT 3
"""
print(f"  SQL:\n    {final_sql.strip()}")
result = execute_select_sqlite(final_sql)

if "error" in result:
    print(f"\n  ❌ {result['phase']}: {result['error']}")
else:
    print(f"\n  ✅ Query succeeded — {result['row_count']} rows\n")
    print("  FINAL ANSWER:")
    print(f"  ┌{'─' * 50}┐")
    print(f"  │ {'Fan ID':<15} {'Total Spend (€)':>17} {'Matches':>14} │")
    print(f"  ├{'─' * 50}┤")
    for row in result["rows"]:
        print(f"  │ {row['fan_id']:<15} {row['total_spend']:>17.2f} {row['matches_attended']:>14} │")
    print(f"  └{'─' * 50}┘")

AGENT TOOL-CALL WALKTHROUGH

  Question: Who are the top 3 fans by total spend, and how many matches did each attend?

  Simulating the agent's ReAct reasoning steps…

────────────────────────────────────────────────────────────
Step 1 → list_tables()
  Agent sees: ['mart_dbt_run_results', 'mart_fan_loyalty', 'mart_match_summary', 'match_events', 'match_metadata', 'merch_purchase', 'retail_purchase']
  Reasoning: mart_fan_loyalty looks like the right table for fan spend data

────────────────────────────────────────────────────────────
Step 2 → describe_table('mart_fan_loyalty')
  Agent sees columns: ['fan_id', 'merch_purchase_count', 'merch_total_spend', 'favourite_merch_item', 'retail_purchase_count', 'retail_total_spend', 'favourite_retail_item', 'favourite_shop', 'total_spend', 'matches_attended', 'first_match_attended', 'last_match_attended', 'last_updated_at']
  Reasoning: total_spend and matches_attended are exactly what I need

──────────────────────────────────────────────────

---
## 5 · ReAct Agent Loop *(optional — requires OpenRouter API key)*

The cell below wires up the SQLite discovery tools as real LangChain tools
and runs a full ReAct agent loop. The agent will:

1. Discover the schema (`list_tables`, `describe_table`)
2. Write and validate SQL (`execute_select_sqlite`)
3. Produce a Markdown answer from the returned rows

> **Requires** `OPENROUTER_API_KEY` set in your `.env` file.
> If the key is missing, the cell skips gracefully.

In [58]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path=REPO_ROOT / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    print("⚠️  OPENROUTER_API_KEY not set — skipping live agent demo.")
    print("   Set it in your .env file and rerun this cell.")
else:
    from langchain.agents import create_agent
    from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
    from langchain_core.tools import tool

    from frontend_app.sql_agent.llm_runtime_config import (
        init_llm_config,
        resolve_agent_model,
    )
    from frontend_app.sql_agent.observability import AgentObservabilityHandler
    from frontend_app.sql_agent.providers import build_chat_model

    init_llm_config()

    # Register tools with the agent. Each tool wraps one of our SQLite helper functions,
    @tool
    def agent_list_tables() -> str:
        """List all tables available in the database."""
        return json.dumps(list_tables())

    @tool
    def agent_describe_table(table: str) -> str:
        """Describe columns of one table (name, type, nullable)."""
        return json.dumps(describe_table(table))

    @tool
    def agent_search_columns(pattern: str) -> str:
        """Search column names across all tables (case-insensitive)."""
        return json.dumps(search_columns(pattern))

    @tool
    def agent_sample_table(table: str, limit: int = 5) -> str:
        """Return a small sample of rows from one table."""
        return json.dumps(sample_table(table, limit))

    @tool
    def agent_execute_select(sql: str) -> str:
        """Execute a validated read-only SELECT query and return JSON rows.

        The SQL is sanitised and validated before execution. Returns
        {"rows": [...], "row_count": N} on success, or
        {"error": "...", "phase": "validation"|"execution"} on failure.
        """
        return json.dumps(execute_select_sqlite(sql))

    all_tools = [
        agent_list_tables,
        agent_describe_table,
        agent_search_columns,
        agent_sample_table,
        agent_execute_select,
    ]

    SYSTEM_PROMPT = """\
You are a careful SQL data analyst for a football club's analytics database.
You answer the user's question by:

1. Discovering what data is available via the supplied tools.
2. Writing a single read-only SELECT statement.
3. Executing it via the `agent_execute_select` tool.
4. Producing a short, clear Markdown answer using the returned rows.

TOOLS:
- `agent_list_tables()` — list every table.
- `agent_describe_table(table)` — columns for one table.
- `agent_search_columns(pattern)` — find columns by name.
- `agent_sample_table(table, limit)` — peek at rows.
- `agent_execute_select(sql)` — the ONLY way to run SQL.

RULES:
- Only SELECT statements. No DDL, no DML, no semicolons.
- Use unqualified table names (no schema prefix).
- On validation/execution errors, fix the SQL and retry.
"""

    agent_model_id = resolve_agent_model()
    llm = build_chat_model(agent_model_id)
    agent_question = (
        "Who are the top 3 fans by total spend, "
        "and how many matches did each attend?"
    )

    print("=" * 60)
    print("REACT AGENT LOOP")
    print("=" * 60)
    print(f"\n  Question: {agent_question}")
    print(f"  Model:    {agent_model_id}")
    print(f"  Tools:    {', '.join(t.name for t in all_tools)}")

    agent = create_agent(
        model=llm, tools=all_tools, system_prompt=SYSTEM_PROMPT
    )

    print(f"\n  Running agent…\n")
    handler = AgentObservabilityHandler()
    state = agent.invoke(
        {"messages": [HumanMessage(content=agent_question)]},
        config={"recursion_limit": 25, "callbacks": [handler]},
    )

    # Display tool call trace and final answer.
    messages = state.get("messages", [])
    print("─" * 60)
    print("TOOL CALL TRACE:")
    print("─" * 60)
    for i, msg in enumerate(messages):
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                args_preview = json.dumps(tc.get("args", {}))[:80]
                print(f"  [{i}] 🤖 → {tc['name']}({args_preview})")
        elif isinstance(msg, ToolMessage):
            content_preview = (
                (msg.content[:100] + "…")
                if len(msg.content) > 100
                else msg.content
            )
            print(f"  [{i}] 🔧 {msg.name} → {content_preview}")

    # Extract and display the final answer.
    final_answer = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and not getattr(
            msg, "tool_calls", None
        ):
            final_answer = (
                msg.content
                if isinstance(msg.content, str)
                else str(msg.content)
            )
            break

    print("\n" + "=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)
    print(f"\n{final_answer}")

2026-04-27 09:39:57.397 | DEBUG    | frontend_app.sql_agent.observability:on_chat_model_start:116 - task=llm_start model=ChatOpenRouter previous=agent_stage_ready next=await_llm_response


REACT AGENT LOOP

  Question: Who are the top 3 fans by total spend, and how many matches did each attend?
  Model:    mistralai/mistral-small-2603
  Tools:    agent_list_tables, agent_describe_table, agent_search_columns, agent_sample_table, agent_execute_select

  Running agent…



2026-04-27 09:39:57.807 | INFO     | frontend_app.sql_agent.observability:on_llm_end:157 - llm_complete model=ChatOpenRouter elapsed_ms=411
2026-04-27 09:39:57.810 | DEBUG    | frontend_app.sql_agent.observability:on_tool_start:219 - task=tool_start tool=agent_list_tables previous=llm_selected_tool next=invoke_tool args={}
2026-04-27 09:39:57.841 | INFO     | frontend_app.sql_agent.observability:on_tool_end:234 - tool_complete tool=agent_list_tables elapsed_ms=31 output=content='[{"name": "mart_dbt_run_results", "type": "table"}, {"name": "mart_fan_
2026-04-27 09:39:57.851 | DEBUG    | frontend_app.sql_agent.observability:on_chat_model_start:116 - task=llm_start model=ChatOpenRouter previous=agent_stage_ready next=await_llm_response
2026-04-27 09:39:58.372 | INFO     | frontend_app.sql_agent.observability:on_llm_end:157 - llm_complete model=ChatOpenRouter elapsed_ms=520
2026-04-27 09:39:58.376 | DEBUG    | frontend_app.sql_agent.observability:on_tool_start:219 - task=tool_start tool=ag

────────────────────────────────────────────────────────────
TOOL CALL TRACE:
────────────────────────────────────────────────────────────
  [1] 🤖 → agent_list_tables({})
  [2] 🔧 agent_list_tables → [{"name": "mart_dbt_run_results", "type": "table"}, {"name": "mart_fan_loyalty", "type": "table"}, {…
  [3] 🤖 → agent_describe_table({"table": "mart_fan_loyalty"})
  [3] 🤖 → agent_describe_table({"table": "merch_purchase"})
  [3] 🤖 → agent_describe_table({"table": "retail_purchase"})
  [4] 🔧 agent_describe_table → {"name": "mart_fan_loyalty", "columns": [{"name": "fan_id", "data_type": "TEXT", "nullable": true, "…
  [5] 🔧 agent_describe_table → {"name": "merch_purchase", "columns": [{"name": "id", "data_type": "INT", "nullable": true, "pk": fa…
  [6] 🔧 agent_describe_table → {"name": "retail_purchase", "columns": [{"name": "id", "data_type": "INTEGER", "nullable": true, "pk…
  [7] 🤖 → agent_execute_select({"sql": "SELECT fan_id, total_spend, matches_attended FROM mart_fan_loyalty ORDE)
  [8

---
## Summary

This notebook demonstrated the complete SQL agent pipeline in a self-contained
SQLite environment with dbt-aligned fan marts:

| Component | What we showed |
|-----------|---------------|
| **Data generation** | Full-season synthetic match-day + retail events from the repo generators |
| **Schema discovery** | `list_tables`, `describe_table`, `search_columns`, `sample_table` — all working against SQLite |
| **SQLite marts** | `mart_fan_loyalty` and `mart_match_summary`, plus helper tables and a placeholder `mart_dbt_run_results` schema |
| **SQL guardrails** | Three-layer validation: fence stripping → schema rewriting → sqlglot AST + regex |
| **Safe execution** | `execute_select_sqlite` with LIMIT 100 safety net |
| **Agent loop** | *(optional)* Full ReAct agent answering a natural language question |

### Key takeaways

1. **Guardrails are backend-agnostic** — the same `_validate_sql` that protects
   Postgres in production works identically against SQLite
2. **Generator-backed marts translate well to SQLite** — `mart_fan_loyalty` and
   `mart_match_summary` can be materialised locally from the same synthetic
   event generators used by the pipeline
3. **The ReAct loop is model-agnostic** — any tool-calling LLM can drive the
   same agent pipeline

`mart_player_season_summary` is intentionally excluded from this notebook.

**Next steps:** See `notebooks/sql-agent.ipynb` for the full production pipeline
with PostgreSQL, the semantic layer, and the repair pass.